In [ ]:
import os
import shutil
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input
from PIL import Image, UnidentifiedImageError
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.utils import class_weight
from scipy.fft import dst

# =========================
# CONFIG
# =========================
RAW_DATASET_DIR = Path("dataset\\wikiart")          # pasta original com as subpastas dos autores
CLEAN_DATASET_DIR = Path("dataset\\dataset_clean")  # pasta para guardar imagens válidas e convertidas
SPLIT_DATASET_DIR = Path("dataset\\dataset_split")  # pasta final com train/val/test

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MIN_WIDTH = 64
MIN_HEIGHT = 64

TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

RANDOM_STATE = 42

assert abs(TRAIN_SIZE + VAL_SIZE + TEST_SIZE - 1.0) < 1e-6, "Os splits têm de somar 1."

Keeping only valid images and organising the dataset:
- `is_valid_image_file` checks if the extension is a valid format.
- `clean_dataset` checks if the files are corrupted, converts to RGB format, filters by size (64/64) and padronises it to .jpg

In [3]:
def is_valid_image_file(file_path):
    return file_path.suffix.lower() in IMG_EXTENSIONS


def clean_dataset(raw_dir, clean_dir, min_width=64, min_height=64):
    clean_dir.mkdir(parents=True, exist_ok=True)

    removed_files = []
    kept_files = []

    class_dirs = [d for d in raw_dir.iterdir() if d.is_dir()]

    for class_dir in class_dirs:
        target_class_dir = clean_dir / class_dir.name
        target_class_dir.mkdir(parents=True, exist_ok=True)

        for file_path in class_dir.iterdir():
            if not file_path.is_file():
                continue

            if not is_valid_image_file(file_path):
                removed_files.append((str(file_path), "invalid"))
                continue

            try:
                with Image.open(file_path) as img:
                    img.verify()

                with Image.open(file_path) as img:
                    img = img.convert("RGB")

                    if img.width < min_width or img.height < min_height:
                        removed_files.append((str(file_path), f"image too small: {img.width}x{img.height}"))
                        continue

                    output_filename = file_path.stem + ".jpg"
                    output_path = target_class_dir / output_filename
                    img.save(output_path, format="JPEG", quality=95)

                    kept_files.append((str(output_path), class_dir.name, img.width, img.height))

            except (UnidentifiedImageError, OSError, Image.DecompressionBombError) as e:
                removed_files.append((str(file_path), f"error opening/processing: {e}"))

    kept_df = pd.DataFrame(kept_files, columns=["filepath", "label", "width", "height"])
    removed_df = pd.DataFrame(removed_files, columns=["filepath", "reason"])

    return kept_df, removed_df

In [4]:
kept_df, removed_df = clean_dataset(
    raw_dir=RAW_DATASET_DIR,
    clean_dir=CLEAN_DATASET_DIR,
    min_width=MIN_WIDTH,
    min_height=MIN_HEIGHT
)

print("Valid:", len(kept_df))
print("Removed:", len(removed_df))


Valid: 13340
Removed: 0


Verify if all authors have an appropriate number of data.

In [5]:
class_counts = kept_df["label"].value_counts().sort_values(ascending=False)

print("\nNumber of images per author:")
print(class_counts)

summary_df = class_counts.reset_index()
summary_df.columns = ["author", "n_images"]

print("\nSummary:")
print(summary_df)


Number of images per author:
label
Vincent_van_Gogh         1322
Nicholas_Roerich         1274
Pierre_Auguste_Renoir     975
Claude_Monet              934
Pyotr_Konchalovsky        644
Camille_Pissarro          621
Albrecht_Durer            580
John_Singer_Sargent       549
Rembrandt                 544
Marc_Chagall              536
Pablo_Picasso             534
Gustave_Dore              528
Boris_Kustodiev           444
Edgar_Degas               428
Paul_Cezanne              406
Ivan_Aivazovsky           404
Martiros_Saryan           403
Eugene_Boudin             389
Childe_Hassam             385
Ilya_Repin                378
Ivan_Shishkin             364
Raphael_Kirchner          362
Salvador_Dali             336
Name: count, dtype: int64

Summary:
                   author  n_images
0        Vincent_van_Gogh      1322
1        Nicholas_Roerich      1274
2   Pierre_Auguste_Renoir       975
3            Claude_Monet       934
4      Pyotr_Konchalovsky       644
5        Camille_Pissa

Solution: Class weights or data augmentation.

In [6]:
def create_splits(df, train_size=0.70, val_size=0.15, test_size=0.15, random_state=42):
    train_df, temp_df = train_test_split(
        df,
        test_size=(1 - train_size),
        stratify=df["label"],
        random_state=random_state
    )

    val_relative = val_size / (val_size + test_size)

    val_df, test_df = train_test_split(
        temp_df,
        test_size=(1 - val_relative),
        stratify=temp_df["label"],
        random_state=random_state
    )

    return train_df, val_df, test_df

In [ ]:
train_df, val_df, test_df = create_splits(
    kept_df,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\ train:")
print(train_df["label"].value_counts())

print(" val:")
print(val_df["label"].value_counts())

print("\ test:")
print(test_df["label"].value_counts())

Train: 9337
Validation: 2001
Test: 2002

Distribuição train:
label
Vincent_van_Gogh         925
Nicholas_Roerich         892
Pierre_Auguste_Renoir    682
Claude_Monet             654
Pyotr_Konchalovsky       451
Camille_Pissarro         435
Albrecht_Durer           406
John_Singer_Sargent      384
Rembrandt                381
Marc_Chagall             375
Pablo_Picasso            374
Gustave_Dore             369
Boris_Kustodiev          311
Edgar_Degas              300
Paul_Cezanne             284
Ivan_Aivazovsky          283
Martiros_Saryan          282
Eugene_Boudin            272
Childe_Hassam            269
Ilya_Repin               265
Ivan_Shishkin            255
Raphael_Kirchner         253
Salvador_Dali            235
Name: count, dtype: int64

Distribuição val:
label
Vincent_van_Gogh         198
Nicholas_Roerich         191
Pierre_Auguste_Renoir    146
Claude_Monet             140
Pyotr_Konchalovsky        96
Camille_Pissarro          93
Albrecht_Durer            87
John_Singer_

In [ ]:
def copy_files(df, split_name, base_dir):
    for _, row in df.iterrows():
        src = row["filepath"]
        label = row["label"]

        # criar pasta destino
        dst_dir = os.path.join(base_dir, split_name, label)
        os.makedirs(dst_dir, exist_ok=True)

        # copiar ficheiro
        dst = os.path.join(dst_dir, os.path.basename(src))

        # Só copia se o ficheiro ainda não existir no destino
        if not os.path.exists(dst):
            shutil.copy(src, dst)

copy_files(train_df, "train", SPLIT_DATASET_DIR)
copy_files(val_df, "val", SPLIT_DATASET_DIR)
copy_files(test_df, "test", SPLIT_DATASET_DIR)

`image_dataset_from_directory` - load and label images from training, validation, and test folders, standardizing them to a uniform $224 \times 224$ pixel resolution in batches of 32.

`preprocess_input function` - scale and normalization of pixel values to match the specific mathematical requirements of EfficentNet sera que vamos usar este???.

`AUTOTUNE` and `prefetch` - CPU prepares the next batch of data while the GPU is still processing the current one, eliminating hardware bottlenecks and significantly speeding up the training process.

In [9]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DATASET_DIR / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DATASET_DIR / "val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DATASET_DIR / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

Found 9337 files belonging to 23 classes.
Found 2001 files belonging to 23 classes.
Found 2002 files belonging to 23 classes.


Inbalance treatment:

In [10]:
# 1. Obter a lista de labels do treino (preservando a ordem do image_dataset_from_directory)
labels_list = train_df['label'].values
unique_classes = np.unique(labels_list)

# 2. Calcular pesos (fórmula: n_samples / (n_classes * np.bincount(y)))
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=labels_list
)

# 3. Criar o dicionário para o Keras 
class_weights = {i: weight for i, weight in enumerate(weights)}

print("Weights:", {class_names[i]: round(class_weights[i], 2) for i in range(5)})

Weights: {'Albrecht_Durer': 1.0, 'Boris_Kustodiev': 1.31, 'Camille_Pissarro': 0.93, 'Childe_Hassam': 1.51, 'Claude_Monet': 0.62}


In [11]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1), # Roda até 10%
    tf.keras.layers.RandomZoom(0.1),     # Zoom in/out de 10%
    tf.keras.layers.RandomContrast(0.1),
])

In [12]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model

# Base congelada
base_model = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
base_model.trainable = False

# Cabeça de classificação
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)
output = layers.Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 410s 1s/step - accuracy: 0.4743 - loss: 1.7728 - val_accuracy: 0.6407 - val_loss: 1.2053
Epoch 2/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 370s 1s/step - accuracy: 0.6608 - loss: 1.1248 - val_accuracy: 0.6837 - val_loss: 1.0386
Epoch 3/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 400s 1s/step - accuracy: 0.7204 - loss: 0.8934 - val_accuracy: 0.7086 - val_loss: 0.9732
Epoch 4/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 368s 1s/step - accuracy: 0.7632 - loss: 0.7373 - val_accuracy: 0.7331 - val_loss: 0.9033
Epoch 5/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 370s 1s/step - accuracy: 0.7895 - loss: 0.6412 - val_accuracy: 0.7346 - val_loss: 0.8917
Epoch 6/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 78672s 270s/step - accuracy: 0.8234 - loss: 0.5302 - val_accuracy: 0.7311 - val_loss: 0.9032
Epoch 7/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 441s 2s/step - accuracy: 0.8466 - loss: 0.4497 - val_accuracy: 0.7521 - val_loss: 0.8664
Epoch 8/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 429s 1s/step - accuracy: 0.8643 - loss: 0.3964 - val_

KeyboardInterrupt: 

In [ ]:
# Descongelar as últimas 30 camadas
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # lr muito mais baixo
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 447s 1s/step - accuracy: 0.7206 - loss: 0.8281 - val_accuracy: 0.6987 - val_loss: 1.0293
Epoch 2/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 79781s 274s/step - accuracy: 0.7712 - loss: 0.6554 - val_accuracy: 0.7056 - val_loss: 0.9970
Epoch 3/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 474s 2s/step - accuracy: 0.8051 - loss: 0.5723 - val_accuracy: 0.7156 - val_loss: 0.9620
Epoch 4/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 438s 1s/step - accuracy: 0.8104 - loss: 0.5375 - val_accuracy: 0.7236 - val_loss: 0.9372
Epoch 5/10
276/292 ━━━━━━━━━━━━━━━━━━━━ 25s 2s/step - accuracy: 0.8246 - loss: 0.5021

In [ ]:
loss, acc = model.evaluate(test_ds)
print(f"Test accuracy: {acc:.4f}")

NameError: name 'model' is not defined